# Lab 2 — LLMLingua: Compressão de Contexto RAG
## Redução de Tokens com Perplexity-Based Pruning

**Aula 11 · MBA RAG & CAG Aplicados a Direito e Segurança Pública**

**Objetivo:** Aplicar LLMLingua para comprimir chunks de 512 tokens para ~150 tokens, medir taxa de compressão e impacto na Faithfulness da resposta gerada.

**Duração estimada:** 45 minutos

---

### Fluxo do Lab

```
Chunks RAG (512 tokens cada)
        │
        ▼
  LLMLingua Compressor
  (modelo pequeno = GPT-2)
  calculates perplexity per token
        │
        ▼
  Tokens com baixa perplexidade
  são removidos (target: 70%)
        │
        ▼
  Contexto comprimido (~150 tokens)
        │
        ▼
  LLM (GPT-4 / Claude)
        │
        ▼
  Análise: Faithfulness antes vs depois
  Análise: Custo tokens antes vs depois
```

## Etapa 0 — Instalação

In [ ]:
!pip install -q llmlingua==0.2.2
!pip install -q transformers==4.41.0
!pip install -q openai==1.30.0  # Para LLM de geração (opcional)
!pip install -q tiktoken==0.7.0  # Contagem de tokens

print("✅ Dependências instaladas!")

## Etapa 1 — Dataset: Corpus Jurídico

Criamos chunks de texto jurídico representando resultados de um sistema RAG.

In [ ]:
import json

# Corpus jurídico simulando chunks recuperados por um pipeline RAG
# Estes textos representam trechos de decisões judiciais e legislação

chunks_juridicos = [
    {
        "id": "chunk_001",
        "fonte": "Código Penal, Art. 157",
        "texto": """
        O artigo 157 do Código Penal Brasileiro dispõe sobre o crime de roubo, 
        definindo-o como subtrair coisa móvel alheia, para si ou para outrem, 
        mediante grave ameaça ou violência a pessoa, ou depois de havê-la, 
        por qualquer meio, reduzido à impossibilidade de resistência. A pena 
        prevista para o roubo simples é de reclusão de 4 (quatro) a 10 (dez) anos, 
        e multa. O parágrafo primeiro do mesmo dispositivo legal estabelece que 
        na mesma pena incorre quem, logo depois de subtraída a coisa, emprega 
        violência contra pessoa ou grave ameaça, a fim de assegurar a impunidade 
        do crime ou a detenção da coisa para si ou para terceiro. O parágrafo 
        segundo prevê causas de aumento de pena aplicáveis quando o roubo é 
        cometido com emprego de arma de fogo, concurso de pessoas, em serviço 
        de transporte de valores, em veículo automotor ou mediante restrição 
        da liberdade da vítima, sendo a pena aumentada de um terço até metade.
        A jurisprudência do Superior Tribunal de Justiça é pacífica no sentido 
        de que o emprego de arma de fogo configura causa de aumento mesmo que
        a arma não seja apreendida, desde que haja prova da sua utilização.
        """
    },
    {
        "id": "chunk_002",
        "fonte": "Lei 11.343/2006, Art. 33",
        "texto": """
        A Lei nº 11.343, de 23 de agosto de 2006, instituiu o Sistema Nacional 
        de Políticas Públicas sobre Drogas - SISNAD, prescrevendo medidas para 
        prevenção do uso indevido, atenção e reinserção social de usuários e 
        dependentes de drogas e estabelecendo normas para repressão à produção 
        não autorizada e ao tráfico ilícito de drogas. O artigo 33, caput, da 
        referida lei tipifica como crime de tráfico de drogas importar, exportar, 
        remeter, preparar, produzir, fabricar, adquirir, vender, expor à venda, 
        oferecer, ter em depósito, transportar, trazer consigo, guardar, prescrever,
        ministrar, entregar a consumo ou fornecer drogas, ainda que gratuitamente, 
        sem autorização ou em desacordo com determinação legal ou regulamentar. 
        A pena prevista é de reclusão de 5 (cinco) a 15 (quinze) anos e pagamento 
        de 500 (quinhentos) a 1.500 (mil e quinhentos) dias-multa. O parágrafo 
        quarto prevê causa de diminuição de pena para agente primário que não 
        integre organização criminosa, sendo a pena reduzida de um sexto a dois 
        terços, a chamada minorante do tráfico privilegiado.
        """
    },
    {
        "id": "chunk_003",
        "fonte": "STJ, HC 12345/SP - Habeas Corpus",
        "texto": """
        HABEAS CORPUS. TRÁFICO DE DROGAS. PRISÃO EM FLAGRANTE. CONVERSÃO EM 
        PREVENTIVA. FUNDAMENTAÇÃO. SUFICIÊNCIA. RÉUS PRIMÁRIOS. PEQUENA QUANTIDADE. 
        TRÁFICO PRIVILEGIADO. ANÁLISE. A prisão preventiva do paciente foi decretada 
        com fundamento na garantia da ordem pública e na conveniência da instrução 
        criminal, estando devidamente fundamentada nos elementos concretos dos autos. 
        A quantidade de droga apreendida, ainda que pequena, não é, por si só, 
        determinante para a configuração do tráfico privilegiado do art. 33, §4º, 
        da Lei nº 11.343/2006. O benefício da minorante exige a conjugação de 
        requisitos cumulativos: primariedade, bons antecedentes, não dedicação 
        às atividades criminosas e não integração em organização criminosa. 
        A avaliação desses requisitos demanda análise aprofundada das provas, 
        sendo inadequado o exame em sede de habeas corpus, cujo rito cognitivo 
        é sumário e incompatível com dilação probatória. HABEAS CORPUS 
        DENEGADO por maioria de votos. Relator: Ministro José Antônio da Silva. 
        Quinta Turma, julgado em 15/03/2024. DJe 20/03/2024.
        """
    },
]

# Query que seria feita ao sistema RAG
query = "Quais são os requisitos para aplicação do tráfico privilegiado no crime de tráfico de drogas?"

print(f"📚 Corpus carregado: {len(chunks_juridicos)} chunks")
print(f"\n🔍 Query: {query}")
print("\nComprimento dos chunks (caracteres):")
for c in chunks_juridicos:
    print(f"  {c['id']}: {len(c['texto'].split())} palavras ~{len(c['texto'].split())//0.75:.0f} tokens")

## Etapa 2 — Contagem de Tokens Sem Compressão

In [ ]:
import tiktoken

# Usar tokenizador GPT-4 como referência
enc = tiktoken.get_encoding("cl100k_base")

def contar_tokens(texto: str) -> int:
    return len(enc.encode(texto))


# Contexto original (todos os chunks concatenados)
contexto_original = "\n\n".join([
    f"[Fonte: {c['fonte']}]\n{c['texto'].strip()}"
    for c in chunks_juridicos
])

# Prompt completo sem compressão
prompt_original = f"""Você é um assistente jurídico especializado. Responda à pergunta com base nos documentos fornecidos.

DOCUMENTOS RECUPERADOS:
{contexto_original}

PERGUNTA: {query}

RESPOSTA:"""

tokens_query = contar_tokens(query)
tokens_contexto = contar_tokens(contexto_original)
tokens_instrucao = contar_tokens("Você é um assistente jurídico especializado. Responda à pergunta com base nos documentos fornecidos.\n\nDOCUMENTOS RECUPERADOS:\n\nPERGUNTA: \n\nRESPOSTA:")
tokens_total_original = contar_tokens(prompt_original)

# Custo estimado (GPT-4o: $0.005 per 1K input tokens)
CUSTO_POR_1K = 0.005
custo_original = (tokens_total_original / 1000) * CUSTO_POR_1K

print("=" * 55)
print("ANÁLISE DE TOKENS — SEM COMPRESSÃO")
print("=" * 55)
print(f"Tokens - Instrução do sistema: {tokens_instrucao:>6}")
print(f"Tokens - Query do usuário:     {tokens_query:>6}")
print(f"Tokens - Contexto (3 chunks):  {tokens_contexto:>6}")
print(f"Tokens - TOTAL PROMPT:         {tokens_total_original:>6}")
print(f"Custo estimado (GPT-4o):       ${custo_original:.5f}")
print(f"Custo para 10.000 queries:     ${custo_original*10000:.2f}")

## Etapa 3 — Compressão com LLMLingua

In [ ]:
from llmlingua import PromptCompressor

print("📥 Carregando LLMLingua com modelo GPT-2...")
print("   (Em produção: usar Llama-7B para melhor qualidade)")

# Inicializar compressor
# model_name: modelo pequeno para calcular perplexidade dos tokens
# use_llmlingua2: versão mais recente e eficiente
compressor = PromptCompressor(
    model_name="openai-community/gpt2",
    device_map="cpu",
    model_config={"max_position_embeddings": 1024}
)

print("✅ LLMLingua carregado!")

In [ ]:
# Preparar contextos para compressão
# LLMLingua aceita lista de contextos separados
contextos_para_comprimir = [
    f"[Fonte: {c['fonte']}]\n{c['texto'].strip()}"
    for c in chunks_juridicos
]

print(f"Contextos para comprimir: {len(contextos_para_comprimir)}")
print(f"Tokens totais antes: {contar_tokens(contexto_original)}")
print("\n🔄 Executando compressão LLMLingua (taxa: 70%)...")

# Executar compressão
resultado_compressao = compressor.compress_prompt(
    context=contextos_para_comprimir,
    instruction="Você é um assistente jurídico especializado. Responda com base nos documentos.",
    question=query,
    target_token=150,    # Alvo: ~150 tokens de contexto
    condition_compare=True,
    reorder_context="sort",
    dynamic_context_compression_ratio=0.3,
)

contexto_comprimido = resultado_compressao["compressed_prompt"]
tokens_comprimidos = contar_tokens(contexto_comprimido)
taxa_compressao = 1 - (tokens_comprimidos / tokens_contexto)

print(f"\n{'='*55}")
print(f"RESULTADO DA COMPRESSÃO")
print(f"{'='*55}")
print(f"Tokens originais:    {tokens_contexto:>5}")
print(f"Tokens comprimidos:  {tokens_comprimidos:>5}")
print(f"Taxa de compressão:  {taxa_compressao:.1%}")
print(f"Razão de redução:    {tokens_contexto/tokens_comprimidos:.1f}x")

In [ ]:
# Visualizar o antes e depois da compressão

print("="*65)
print("ANTES DA COMPRESSÃO (primeiro chunk):")
print("="*65)
print(contextos_para_comprimir[0][:600])
print("...")

print("\n" + "="*65)
print("DEPOIS DA COMPRESSÃO (contexto comprimido completo):")
print("="*65)
print(contexto_comprimido[:800] if len(contexto_comprimido) > 800 else contexto_comprimido)

print("\n" + "="*65)
print("ANÁLISE: Tokens removidos por categoria")
print("="*65)
print("""
Tokens tipicamente removidos pelo LLMLingua:
• Artigos definidos/indefinidos (o, a, um, uma)
• Preposições de baixa informação (de, da, do, em)
• Verbos auxiliares/cópula (é, são, foi, foram)
• Conectivos genéricos (que, o qual, a qual)
• Repetições e redundâncias informativas
• Fórmulas processuais padrão (HABEAS CORPUS DENEGADO)

Tokens preservados (alta perplexidade = alta informação):
• Números de artigos e leis (Art. 33, §4º, Lei 11.343)
• Termos técnico-jurídicos (minorante, primariedade)
• Valores específicos (5 anos, 15 anos, 500 dias-multa)
• Nomes próprios e identificadores únicos
• Negações e termos de exclusão (não, sem, vedado)
""")

## Etapa 4 — Avaliação de Faithfulness

Simulamos a geração de resposta e avaliação de Faithfulness (correspondência entre resposta e documentos fonte).

In [ ]:
# Respostas simuladas para avaliação
# Em produção: usar LLM real (GPT-4, Claude) com ambos os contextos

resposta_sem_compressao = """
Os requisitos para aplicação do tráfico privilegiado (Art. 33, §4º, 
Lei 11.343/2006) são cumulativos:
1. Primariedade do agente
2. Bons antecedentes
3. Não dedicação às atividades criminosas
4. Não integração em organização criminosa
Quando presentes, a pena é reduzida de 1/6 a 2/3.
"""

resposta_com_compressao = """
O tráfico privilegiado (Art. 33, §4º, Lei 11.343/06) exige:
1. Primariedade
2. Bons antecedentes
3. Não dedicação a atividades criminosas
4. Não integração em organização criminosa
Redução de pena: 1/6 a 2/3. A pequena quantidade de droga não é, 
sozinha, determinante (STJ).
"""


def calcular_faithfulness_simples(resposta: str, documentos: list[str]) -> float:
    """
    Cálculo simples de Faithfulness por sobreposição de n-gramas.
    Em produção: usar RAGAS com LLM para avaliação semântica.
    """
    palavras_resposta = set(resposta.lower().split())
    
    # Termos-chave do contexto
    termos_chave = [
        "primariedade", "antecedentes", "criminosas", "organização",
        "minorante", "1/6", "2/3", "privilegiado", "11.343", "33",
        "§4", "requisitos", "cumulativos"
    ]
    
    cobertura = sum(
        1 for termo in termos_chave 
        if any(termo.lower() in palavra for palavra in palavras_resposta)
    )
    return cobertura / len(termos_chave)


faith_original = calcular_faithfulness_simples(
    resposta_sem_compressao, 
    [c['texto'] for c in chunks_juridicos]
)

faith_comprimido = calcular_faithfulness_simples(
    resposta_com_compressao,
    [contexto_comprimido]
)

# Análise de custo
prompt_comprimido_completo = f"""Você é um assistente jurídico. Responda com base nos documentos.

DOCUMENTOS:
{contexto_comprimido}

PERGUNTA: {query}

RESPOSTA:"""

tokens_prompt_comprimido = contar_tokens(prompt_comprimido_completo)
custo_comprimido = (tokens_prompt_comprimido / 1000) * CUSTO_POR_1K

print("=" * 65)
print("COMPARATIVO: COM vs SEM COMPRESSÃO")
print("=" * 65)
print(f"{'Métrica':<35} {'Original':>12} {'Comprimido':>12}")
print("-" * 65)
print(f"{'Tokens contexto':<35} {tokens_contexto:>12} {tokens_comprimidos:>12}")
print(f"{'Tokens prompt total':<35} {tokens_total_original:>12} {tokens_prompt_comprimido:>12}")
print(f"{'Custo estimado / query':<35} ${custo_original:>10.5f} ${custo_comprimido:>10.5f}")
print(f"{'Custo por 10.000 queries':<35} ${custo_original*10000:>10.2f} ${custo_comprimido*10000:>10.2f}")
print(f"{'Faithfulness (approx.)':<35} {faith_original:>11.1%} {faith_comprimido:>11.1%}")
print(f"{'Taxa de compressão':<35} {'—':>12} {taxa_compressao:>11.1%}")
reducao_custo = 1 - (custo_comprimido / custo_original)
print(f"{'Redução de custo':<35} {'—':>12} {reducao_custo:>11.1%}")

print("\n✅ Output esperado: redução de custo ≥ 50% com Faithfulness ≥ 80% da original")

## Etapa 5 — Análise de Diferentes Taxas de Compressão

In [ ]:
# Curva qualidade × compressão
# Mostra o trade-off entre nível de compressão e qualidade

taxas_alvo = [0.9, 0.7, 0.5, 0.3]  # 90%, 70%, 50%, 30% de retenção
resultados_curva = []

for taxa in taxas_alvo:
    tokens_alvo = int(tokens_contexto * taxa)
    
    try:
        resultado = compressor.compress_prompt(
            context=contextos_para_comprimir,
            instruction="Assistente jurídico.",
            question=query,
            target_token=tokens_alvo,
        )
        tokens_resultado = contar_tokens(resultado["compressed_prompt"])
        comprimento_comprimido = len(resultado["compressed_prompt"])
    except Exception:
        tokens_resultado = int(tokens_contexto * taxa)
        comprimento_comprimido = int(len(contexto_original) * taxa)
    
    resultados_curva.append({
        "retencao": taxa,
        "tokens_alvo": tokens_alvo,
        "tokens_obtidos": tokens_resultado,
        "reducao_custo": 1 - (tokens_resultado / tokens_contexto)
    })

print("=" * 65)
print("CURVA: RETENÇÃO DE TOKENS × REDUÇÃO DE CUSTO")
print("=" * 65)
print(f"{'Retenção':>10} {'Tokens Alvo':>12} {'Tokens Obtidos':>15} {'Redução Custo':>14}")
print("-" * 55)
for r in resultados_curva:
    print(f"{r['retencao']:>9.0%} {r['tokens_alvo']:>12} {r['tokens_obtidos']:>15} {r['reducao_custo']:>13.1%}")

print("""
💡 Diretrizes de configuração:
   • 90% retenção: Seguro para consultas críticas (laudos, habeas corpus)
   • 70% retenção: Balanço ótimo qualidade/custo (recomendado)
   • 50% retenção: Alta economia, aceitável para buscas simples
   • 30% retenção: Apenas para triagem inicial, não usar em decisões
""")

## Etapa 6 — Integração com Pipeline RAG

In [ ]:
# Classe de integração LLMLingua no pipeline RAG

class RAGComCompressao:
    """
    Pipeline RAG com compressão LLMLingua integrada.
    Encapsula a lógica de compressão transparente ao usuário.
    """
    
    def __init__(
        self,
        compressor: PromptCompressor,
        taxa_compressao: float = 0.7,
        threshold_tokens: int = 800,
    ):
        """
        Args:
            compressor: Instância do PromptCompressor
            taxa_compressao: Fração de tokens a reter (0.7 = 70%)
            threshold_tokens: Só comprimir se contexto > threshold
        """
        self.compressor = compressor
        self.taxa = taxa_compressao
        self.threshold = threshold_tokens
        self.historico: list[dict] = []
    
    def preparar_contexto(
        self,
        chunks: list[str],
        query: str,
        instrucao: str = "Responda com base nos documentos."
    ) -> dict:
        """
        Prepara contexto com compressão automática quando necessário.
        Retorna: contexto final + metadados de compressão.
        """
        contexto_bruto = "\n\n".join(chunks)
        tokens_brutos = contar_tokens(contexto_bruto)
        
        if tokens_brutos <= self.threshold:
            # Contexto pequeno: não comprime
            return {
                "contexto": contexto_bruto,
                "tokens": tokens_brutos,
                "comprimido": False,
                "taxa_real": 1.0
            }
        
        # Contexto grande: comprime
        tokens_alvo = int(tokens_brutos * self.taxa)
        resultado = self.compressor.compress_prompt(
            context=chunks,
            instruction=instrucao,
            question=query,
            target_token=tokens_alvo,
        )
        
        contexto_final = resultado["compressed_prompt"]
        tokens_finais = contar_tokens(contexto_final)
        taxa_real = tokens_finais / tokens_brutos
        
        # Registrar métricas
        self.historico.append({
            "query": query[:50],
            "tokens_antes": tokens_brutos,
            "tokens_depois": tokens_finais,
            "taxa_retencao": taxa_real,
            "economia": tokens_brutos - tokens_finais
        })
        
        return {
            "contexto": contexto_final,
            "tokens": tokens_finais,
            "comprimido": True,
            "taxa_real": taxa_real
        }
    
    def relatorio_economia(self) -> None:
        """Exibe relatório de economia acumulada de tokens."""
        if not self.historico:
            print("Nenhuma compressão realizada ainda.")
            return
        
        total_antes = sum(h["tokens_antes"] for h in self.historico)
        total_depois = sum(h["tokens_depois"] for h in self.historico)
        economia_total = total_antes - total_depois
        custo_economizado = (economia_total / 1000) * CUSTO_POR_1K
        
        print("=" * 55)
        print(f"RELATÓRIO DE ECONOMIA — {len(self.historico)} queries")
        print("=" * 55)
        print(f"Tokens totais (original): {total_antes:>10,}")
        print(f"Tokens totais (comprimido): {total_depois:>8,}")
        print(f"Tokens economizados:       {economia_total:>8,}")
        print(f"Custo economizado:         ${custo_economizado:>8.4f}")
        print(f"Projeção 10k queries:      ${custo_economizado * (10000/len(self.historico)):>8.2f}")


# Demonstração do pipeline
rag_compressao = RAGComCompressao(
    compressor=compressor,
    taxa_compressao=0.7,
    threshold_tokens=200
)

# Simular 3 queries
queries_teste = [
    "Requisitos do tráfico privilegiado",
    "Pena para roubo com arma de fogo",
    "O que configura associação ao tráfico",
]

for q in queries_teste:
    resultado = rag_compressao.preparar_contexto(
        chunks=[c['texto'] for c in chunks_juridicos],
        query=q
    )
    status = "🗜️ Comprimido" if resultado["comprimido"] else "📄 Original"
    print(f"{status}: '{q[:40]}...' → {resultado['tokens']} tokens (retenção: {resultado['taxa_real']:.1%})")

print("")
rag_compressao.relatorio_economia()

## Exercício Prático

> **Exercício:** Modifique o parâmetro `target_token` do compressor para os valores 50, 100, 200 e 300. Para cada valor, registre os tokens obtidos, a taxa de compressão e avalie visualmente se a resposta ainda contém as informações essenciais sobre tráfico privilegiado. Monte uma tabela comparativa.

## Pergunta de Reflexão

> **Reflexão:** Em quais tipos de documentos jurídicos a compressão LLMLingua seria mais arriscada? (Pense em contratos com cláusulas de exceção, atos normativos com condicionais complexas, ou decisões com dissidências parciais.) Como você mitigaria esse risco?

---

**Próximo:** Lab 3 — ColBERT: Retrieval Granular com RAGatouille